In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

# Ruta base
base_dir = Path("results")

alg_dirs = ["MOEAD", "NSGA2", "SMSEMOA"]
csv_paths = [base_dir / alg / "runs_summary.csv" for alg in alg_dirs]

# ✅ Columnas objetivo a normalizar (ajusta si quieres usar *_mean_feas)
f1_col = "f1_min_feas"
f2_col = "f2_min_feas"

# 1️⃣ Leer todos los CSV
dfs = []
for path in csv_paths:
    df = pd.read_csv(path)
    df["_source_csv"] = path
    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)

# 2️⃣ Min y max globales para f1 y f2
global_f1_min = all_df[f1_col].min()
global_f1_max = all_df[f1_col].max()
global_f2_min = all_df[f2_col].min()
global_f2_max = all_df[f2_col].max()

print(f"{f1_col} global min = {global_f1_min:.6f}")
print(f"{f1_col} global max = {global_f1_max:.6f}")
print(f"{f2_col} global min = {global_f2_min:.6f}")
print(f"{f2_col} global max = {global_f2_max:.6f}")

# 3️⃣ Normalización Min-Max (robusta si max == min)
f1_denom = global_f1_max - global_f1_min
f2_denom = global_f2_max - global_f2_min

all_df["f1_norm"] = np.where(
    f1_denom == 0,
    0.0,
    (all_df[f1_col] - global_f1_min) / f1_denom
)

all_df["f2_norm"] = np.where(
    f2_denom == 0,
    0.0,
    (all_df[f2_col] - global_f2_min) / f2_denom
)

# (Opcional) forzar rango [0, 1] por estabilidad numérica
all_df["f1_norm"] = all_df["f1_norm"].clip(0, 1)
all_df["f2_norm"] = all_df["f2_norm"].clip(0, 1)

# 4️⃣ Guardar cada CSV con las nuevas columnas
for path in csv_paths:
    df_alg = all_df[all_df["_source_csv"] == path].drop(columns="_source_csv")
    df_alg.to_csv(path, index=False)
    print(f"✔ Guardado: {path}")


f1_min_feas global min = -0.982929
f1_min_feas global max = -0.638360
f2_min_feas global min = 1.000000
f2_min_feas global max = 6.000000
✔ Guardado: results/MOEAD/runs_summary.csv
✔ Guardado: results/NSGA2/runs_summary.csv
✔ Guardado: results/SMSEMOA/runs_summary.csv


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp
import matplotlib.pyplot as plt
from pathlib import Path

base = Path("results")
alg_dirs = ["MOEAD", "NSGA2", "SMSEMOA"]

dfs = []
for d in alg_dirs:
    df = pd.read_csv(base / d / "runs_summary.csv")
    dfs.append(df)

metrics_df = pd.concat(dfs, ignore_index=True)


In [3]:
metric = "hypervolume_norm"

pivot = metrics_df.pivot_table(
    index="run_id",          # bloque
    columns="algorithm",     # tratamientos
    values=metric
)
pivot


algorithm,NSGA2,moead,smsemoa
run_id,,,
1,0.953093,0.402397,0.944197
2,0.931133,0.826179,0.962995
3,0.971334,0.000000,0.946606
4,0.952554,0.703541,0.928544
5,1.000000,0.778453,0.937674
6,0.973311,0.581500,0.967914
7,0.979024,0.784162,0.962816
8,0.961182,0.674672,0.964183
9,0.959071,0.755097,0.970699


In [4]:
friedman_stat, p_value = friedmanchisquare(
    *[pivot[col].values for col in pivot.columns]
)

print(f"Friedman χ² = {friedman_stat:.4f}, p-value = {p_value:.6f}")


Friedman χ² = 15.2000, p-value = 0.000500


In [5]:
import scikit_posthocs as sp
import numpy as np

# Matriz en formato correcto (filas = bloques, columnas = algoritmos)
data_matrix = pivot.values  

nemenyi = sp.posthoc_nemenyi_friedman(data_matrix)

nemenyi.columns = pivot.columns
nemenyi.index = pivot.columns

print(nemenyi)


algorithm     NSGA2     moead   smsemoa
algorithm                              
NSGA2      1.000000  0.001009  0.887683
moead      0.001009  1.000000  0.004967
smsemoa    0.887683  0.004967  1.000000


In [6]:
ranks = pivot.rank(axis=1, ascending=False)
print(ranks.mean())


algorithm
NSGA2      1.4
moead      3.0
smsemoa    1.6
dtype: float64
